### ICA - Independent Component Analysis

In [15]:
import numpy as np
from sklearn.decomposition import FastICA

# Step-1: Create 2 simple independent signals
S=np.array([[1,1],[2,-1],[3,1],[4,1]])

# Step 2: Mix them
A=np.array([[1,2],[3,4]])
X=S@A.T  #Observed Mixed data

# Step 3: Apply ICA
ica=FastICA(n_components=2, random_state=0)
U=ica.fit_transform(X)
print("Original Sources:\n",S)
print("Recovered ources (approx):\n",U)

Original Sources:
 [[ 1  1]
 [ 2 -1]
 [ 3  1]
 [ 4  1]]
Recovered ources (approx):
 [[-0.81984894  1.42903234]
 [-1.10504234 -1.33374714]
 [ 0.60598672  0.24769239]
 [ 1.31890456 -0.34297759]]


### Implementation of Autoencoders

In [16]:
#Implementation of Autoencoders

from sklearn.neural_network import MLPRegressor

#Step1: Create simple data(4 features)
X = np.array([[1,2,3,4],[2,3,4,5],[3,4,5,6],[4,5,6,7]],dtype=float)

#Step2: Create Autoencoder (4->2->4)
autoencoder = MLPRegressor(hidden_layer_sizes=(2,),activation="relu",max_iter=5000,random_state=54)

#Step3: Train to reproduce input
autoencoder.fit(X,X)

#Step4: Reconstruct
X_reconstructed = autoencoder.predict(X)

#Step 5: Print 2D hidden representation (the "4->2->" parameter)
W = autoencoder.coefs_[0] # (4 X 2) input->hidden weight
b = autoencoder.intercepts_[0] # (2, ) hidden biases
hidden_2d = np.maximum(0,X@W+b) #ReLU: max(0,...)

print("Original(4 features):\n",X)
print("\nHidden(2 features) ", np.round(hidden_2d,2))
print("\nReconstructed: \n",np.round(X_reconstructed))

Original(4 features):
 [[1. 2. 3. 4.]
 [2. 3. 4. 5.]
 [3. 4. 5. 6.]
 [4. 5. 6. 7.]]

Hidden(2 features)  [[3.02 4.78]
 [3.99 6.48]
 [4.95 8.18]
 [5.92 9.88]]

Reconstructed: 
 [[2. 3. 3. 4.]
 [2. 3. 4. 5.]
 [3. 4. 5. 6.]
 [3. 5. 6. 7.]]


### Implementation of Active Learning

In [17]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

#Create simple dataset (fix parameters properly)
x,y = make_classification(n_samples=100,n_features=2,n_redundant=0,n_informative=2,random_state=54)

#Start with 10 labelled samples
labelled = np.arange(10)
unlabelled = np.arange(10,100)
model = LogisticRegression(solver="liblinear")

for z in range(5):
    model.fit(x[labelled],y[labelled])
    probs = model.predict_proba(x[unlabelled])[:,1]
    
    #pick most unceratin sample
    query = np.argmin(np.abs(probs-0.5))
    
    #move that sample to labelled set
    labelled = np.append(labelled,unlabelled[query])
    unlabelled = np.delete(unlabelled,query)
    
    print("Iteration: ",z+1, "|Labelled size: ", len(labelled))

Iteration:  1 |Labelled size:  11
Iteration:  2 |Labelled size:  12
Iteration:  3 |Labelled size:  13
Iteration:  4 |Labelled size:  14
Iteration:  5 |Labelled size:  15


### RBF - Radial Basis Function

In [18]:
#Simple 1D input data
x,y = np.array([[1],[2],[3],[4]]), np.array([0,0,1,1])

#Choose 2 fixed RBF centers
centers = np.array([[1],[4]])
sigma = 1.0

#Gaussian RBF function
def rbf(x,c):
    return np.exp(-np.linalg.norm(x-c)**2/(2*sigma**2))

#Build hidden layer output
H = np.array([[rbf(X,c) for c in centers] for X in x])

#Solve for weights using simple linear equation
w = np.linalg.pinv(H)@y #Moore-Penrose inverse

#Predict
y_pred = H @ w
y_pred_class = (y_pred>0.5).astype(int)

print("Predicted: ",y_pred_class)

Predicted:  [0 0 1 1]
